In [1]:
from reachy_sdk import ReachySDK

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Connect to Reachy. If you're working directly on the robot, use

```python
host='localhost'
```

If not, just get Reachy's IP address with

```bash
$ ifconfig
```

and replace the host argument zith the IP address.

In [2]:
reachy = ReachySDK(host='10.22.129.133')

Check if you see the five joints of the head.

In [3]:
reachy.head

<Head joints=<Holder
	<Joint name="neck_roll" pos="10.66" mode="stiff">
	<Joint name="neck_pitch" pos="41.21" mode="stiff">
	<Joint name="neck_yaw" pos="-19.94" mode="stiff">
	<Joint name="l_antenna" pos="1.17" mode="stiff">
	<Joint name="r_antenna" pos="-118.07" mode="stiff">
>>

In [4]:
for joint_name in dir(reachy.head.joints):
    if not joint_name.startswith("_"):
        joint = getattr(reachy.head.joints, joint_name)
        print(joint_name, joint)

items <bound method DeviceHolder.items of <Holder
	<Joint name="neck_roll" pos="10.60" mode="stiff">
	<Joint name="neck_pitch" pos="41.26" mode="stiff">
	<Joint name="neck_yaw" pos="-19.91" mode="stiff">
	<Joint name="l_antenna" pos="1.17" mode="stiff">
	<Joint name="r_antenna" pos="-118.07" mode="stiff">
>>
keys <bound method DeviceHolder.keys of <Holder
	<Joint name="neck_roll" pos="10.60" mode="stiff">
	<Joint name="neck_pitch" pos="41.26" mode="stiff">
	<Joint name="neck_yaw" pos="-19.91" mode="stiff">
	<Joint name="l_antenna" pos="1.17" mode="stiff">
	<Joint name="r_antenna" pos="-118.07" mode="stiff">
>>
l_antenna <Joint name="l_antenna" pos="1.17" mode="stiff">
neck_pitch <Joint name="neck_pitch" pos="41.26" mode="stiff">
neck_roll <Joint name="neck_roll" pos="10.60" mode="stiff">
neck_yaw <Joint name="neck_yaw" pos="-19.91" mode="stiff">
r_antenna <Joint name="r_antenna" pos="-118.07" mode="stiff">
values <bound method DeviceHolder.values of <Holder
	<Joint name="neck_roll" pos

Use look_at to see if the zero has been actually done.

## Checking Orbita

Make Reachy look forward.

In [5]:
reachy.turn_on("head")

In [6]:
reachy.head.joints.neck_yaw.goal_position = -20

In [7]:
reachy.head.look_at(
    x=1.0,
    y=0.5,
    z=1.2,
    duration=2
)

In [8]:
reachy.head.look_at(
    x=0.5,
    y=0,
    z=0,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

If this didn't work, it's likely that the zeros of Orbita have not been set. To do that use [orbita_zero.py](https://github.com/pollen-robotics/reachy_controllers/blob/master/setup/orbita_zero.py).

Next, reproduce the look_at sequence of the [documentation](https://pollen-robotics.github.io/reachy-2021-docs/sdk/first-moves/head/#orbita-look_at-method).

In [ ]:
import time

look_right = reachy.head.look_at(
    x=0.5,
    y=-0.5,
    z=0.1,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

time.sleep(0.5)

look_down = reachy.head.look_at(
    x=0.5,
    y=0,
    z=-0.4,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

time.sleep(0.5)

look_left = reachy.head.look_at(
    x=0.5,
    y=0.3,
    z=-0.3,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

time.sleep(0.5)

look_front = reachy.head.look_at(
    x=0.5,
    y=0,
    z=0,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

Make the head follow the end-effector. It is a good opportunity to check the forward kinematics of each arm as well.

**Press the square button 'interrupt the kernel' when you want to stop this.**

With the right arm:

In [ ]:
try:
    x, y, z = reachy.r_arm.forward_kinematics()[:3,-1] # We want the translation part of Reachy's pose matrix
    reachy.head.look_at(x=x, y=y, z=z-0.05, duration=1.0) # There is a 5cm offset on the z axis

    time.sleep(0.5)

    while True:
        x, y, z = reachy.r_arm.forward_kinematics()[:3,-1]
        gp_dic = reachy.head._look_at(x, y, z - 0.05)
        reachy.head.neck_roll.goal_position = gp_dic[reachy.head.neck_roll]
        reachy.head.neck_pitch.goal_position = gp_dic[reachy.head.neck_pitch]
        reachy.head.neck_yaw.goal_position = gp_dic[reachy.head.neck_yaw]
        time.sleep(0.01)
except AttributeError:
    print('Reachy has no right arm!')

In [ ]:
reachy.head.joints.neck_roll.goal_position = 0
reachy.head.joints.neck_pitch.goal_position = 0
reachy.head.joints.neck_yaw.goal_position = 0

and turn off the motors.

In [9]:
reachy.turn_off('head')